In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/)
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch  # import torch first to avoid circular import
from kaggle_secrets import UserSecretsClient
import wandb

# Get API key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import lightgbm as lgb
from sentence_transformers import CrossEncoder, InputExample
import wandb

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# *MILESTONE 3 

In [ ]:
!pip install faiss-gpu sentence-transformers -q

In [8]:
import pandas as pd 
import numpy as np 
import faiss 
import pickle
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 
print("Creating knowledge base") 
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 
print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=True) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)
print("Saving knowledge base and FAISS index...")
with open('kb.pkl', 'wb') as f:
    pickle.dump(kb, f)
faiss.write_index(index, 'kb_index.faiss')
print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Saving knowledge base and FAISS index...
Knowledge base successfully created



# Zero-shot classifier for Q1, Q2, Q6zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") row_150 = train.iloc[150] prompt_150 = str(row_150['prompt']) labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] ans_150 = str(row_150[row_150['answer']])

# Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [2]:
import pandas as pd
from transformers import pipeline

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Initialize the zero-shot classifier
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 

# Retrieve data for row index 150
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [
    str(row_150['A']), 
    str(row_150['B']), 
    str(row_150['C']), 
    str(row_150['D']), 
    str(row_150['E'])
] 
ans_150 = str(row_150[row_150['answer']])

# Run the zero-shot classifier
result = zs(prompt_150, candidate_labels=labels_150)

# Extract the score for the ground-truth correct option
correct_score = None
for label, score in zip(result['labels'], result['scores']):
    if label == ans_150:
        correct_score = score
        break

print(f"Correct Option Probability Score: {correct_score:.6f}")
print(f"Rounded to 3 decimal points: {correct_score:.3f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Correct Option Probability Score: 0.384419
Rounded to 3 decimal points: 0.384


# Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [10]:
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Load the saved knowledge base and FAISS index
with open('kb.pkl', 'rb') as f:
    kb = pickle.load(f)
index = faiss.read_index('kb_index.faiss')

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Get prompt and target document for index 150
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
target_doc = kb[150]

# Embed the query prompt
query_embedding = model.encode([prompt_150], show_progress_bar=False)

# Query the FAISS index for top k=10
k = 10
distances, indices = index.search(query_embedding, k)

# Extract lists
indices = indices[0]
distances = distances[0]

# Display the ranks
correct_rank = None
for rank, idx in enumerate(indices, start=1):
    is_correct = (idx == 150)
    print(f"Rank {rank}: KB Index {idx}, Distance: {distances[rank-1]:.6f}, Matches Target: {is_correct}")
    if is_correct:
        correct_rank = rank

print(f"\nResult: True correct document rank is {correct_rank}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rank 1: KB Index 663, Distance: 0.263943, Matches Target: False
Rank 2: KB Index 1701, Distance: 0.263943, Matches Target: False
Rank 3: KB Index 1269, Distance: 0.266428, Matches Target: False
Rank 4: KB Index 1532, Distance: 0.266428, Matches Target: False
Rank 5: KB Index 576, Distance: 0.268622, Matches Target: False
Rank 6: KB Index 847, Distance: 0.269936, Matches Target: False
Rank 7: KB Index 1693, Distance: 0.269936, Matches Target: False
Rank 8: KB Index 1906, Distance: 0.269936, Matches Target: False
Rank 9: KB Index 168, Distance: 0.283161, Matches Target: False
Rank 10: KB Index 150, Distance: 0.287146, Matches Target: True

Result: True correct document rank is 10


# Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
# In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. 
# To fix this we use a 2 stage pipeline:
# 1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
# 2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

# A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

# For this we part the prompt and the context and ask it to predict a score Cross Encoders (Hugging Face)


# Code to use a Cross-Encodercross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunkspairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairsce_scores = cross_encoder.predict(pairs) # Get the score of each pair

# Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [12]:
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Load the saved knowledge base and FAISS index
with open('kb.pkl', 'rb') as f:
    kb = pickle.load(f)
index = faiss.read_index('kb_index.faiss')

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Get prompt and target document for index 150
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])

# Embed the prompt and query the FAISS index (retrieve top 10)
query_embedding = model.encode([prompt_150], show_progress_bar=False)
k = 10
distances, indices = index.search(query_embedding, k)
retrieved_indices = indices[0]

# Load the Cross-Encoder model
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Prepare the prompt-context pairs
docs_10 = [kb[idx] for idx in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]

# Predict semantic similarity scores
ce_scores = cross_encoder.predict(pairs)

# Combine and sort results by score descending
results = []
for i, idx in enumerate(retrieved_indices):
    results.append({
        'kb_index': idx,
        'doc_text': docs_10[i],
        'faiss_rank': i + 1,
        'faiss_distance': distances[0][i],
        'ce_score': ce_scores[i]
    })

sorted_results = sorted(results, key=lambda x: x['ce_score'], reverse=True)

# Display reranking results
for rank, res in enumerate(sorted_results, start=1):
    is_correct = (res['kb_index'] == 150)
    print(f"Rank {rank}: KB Index {res['kb_index']}, CE Score: {res['ce_score']:.6f}, Original FAISS Rank: {res['faiss_rank']}, Matches Target: {is_correct}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Rank 1: KB Index 150, CE Score: 4.758512, Original FAISS Rank: 10, Matches Target: True
Rank 2: KB Index 847, CE Score: 4.752577, Original FAISS Rank: 6, Matches Target: False
Rank 3: KB Index 1693, CE Score: 4.752577, Original FAISS Rank: 7, Matches Target: False
Rank 4: KB Index 1906, CE Score: 4.752577, Original FAISS Rank: 8, Matches Target: False
Rank 5: KB Index 1269, CE Score: 4.737523, Original FAISS Rank: 3, Matches Target: False
Rank 6: KB Index 1532, CE Score: 4.737523, Original FAISS Rank: 4, Matches Target: False
Rank 7: KB Index 168, CE Score: 4.707248, Original FAISS Rank: 9, Matches Target: False
Rank 8: KB Index 576, CE Score: 4.687043, Original FAISS Rank: 5, Matches Target: False
Rank 9: KB Index 663, CE Score: 4.660232, Original FAISS Rank: 1, Matches Target: False
Rank 10: KB Index 1701, CE Score: 4.660232, Original FAISS Rank: 2, Matches Target: False


# Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [14]:
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Load the saved knowledge base and FAISS index
with open('kb.pkl', 'rb') as f:
    kb = pickle.load(f)
index = faiss.read_index('kb_index.faiss')

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Get prompt for row index 42
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

# Embed prompt and query FAISS for top k=5
query_embedding = model.encode([prompt_42], show_progress_bar=False)
k = 5
distances, indices = index.search(query_embedding, k)
retrieved_indices = indices[0]

# Retrieve and concatenate the top 5 documents
docs_5 = [kb[idx] for idx in retrieved_indices]
concatenated_docs = " ".join(docs_5)

# Format the final input string
final_string = f"Context: {concatenated_docs} Question: {prompt_42}"

# Load the bert-base-uncased tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenize and encode
tokens = tokenizer.tokenize(final_string)
input_ids_with_special = tokenizer.encode(final_string, add_special_tokens=True)
input_ids_no_special = tokenizer.encode(final_string, add_special_tokens=False)

print(f"Tokens (tokenize): {len(tokens)}")
print(f"Tokens (encode, with special [CLS]/[SEP]): {len(input_ids_with_special)}")
print(f"Tokens (encode, without special): {len(input_ids_no_special)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokens (tokenize): 214
Tokens (encode, with special [CLS]/[SEP]): 216
Tokens (encode, without special): 214


# Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [16]:
import pickle
import pandas as pd
from transformers import pipeline

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Load the saved knowledge base
with open('kb.pkl', 'rb') as f:
    kb = pickle.load(f)

# Load zero-shot classifier
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Get target values for row index 150
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
true_document = kb[150]
labels_150 = [
    str(row_150['A']), 
    str(row_150['B']), 
    str(row_150['C']), 
    str(row_150['D']), 
    str(row_150['E'])
]
ans_150 = str(row_150[row_150['answer']])

# Create the augmented RAG string
rag_string = f"Context: {true_document} Question: {prompt_150}"

# Run classification on the RAG string
result = zs(rag_string, candidate_labels=labels_150)

# Find the score of the correct answer
correct_score = None
for label, score in zip(result['labels'], result['scores']):
    if label == ans_150:
        correct_score = score
        break

print(f"Correct answer score: {correct_score:.6f}")
print(f"Rounded to 3 decimal places: {correct_score:.3f}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Correct answer score: 0.989426
Rounded to 3 decimal places: 0.989



# Concept: The Danger of Bad Retrieval (Adversarial RAG) The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.



# Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).*



In [18]:
import pickle
import pandas as pd
from transformers import pipeline

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Load the saved knowledge base
with open('kb.pkl', 'rb') as f:
    kb = pickle.load(f)

# Load zero-shot classifier
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Get prompt, adversarial context (index 999), and candidate labels for row index 150
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
adversarial_document = kb[999]
labels_150 = [
    str(row_150['A']), 
    str(row_150['B']), 
    str(row_150['C']), 
    str(row_150['D']), 
    str(row_150['E'])
]
ans_150 = str(row_150[row_150['answer']])

# Create the adversarial RAG string
adversarial_rag_string = f"Context: {adversarial_document} Question: {prompt_150}"

# Run classification
result = zs(adversarial_rag_string, candidate_labels=labels_150)

# Find the score of the correct answer
correct_score = None
for label, score in zip(result['labels'], result['scores']):
    if label == ans_150:
        correct_score = score
        break

print(f"Adversarial correct option score: {correct_score:.6f}")
print(f"Rounded to 3 decimal places: {correct_score:.3f}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Adversarial correct option score: 0.528948
Rounded to 3 decimal places: 0.529


# In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

# Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [19]:
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Load the saved knowledge base and FAISS index
with open('kb.pkl', 'rb') as f:
    kb = pickle.load(f)
index = faiss.read_index('kb_index.faiss')

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

num_rows = 100
hits = 0

for i in range(num_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    correct_option_text = str(row[row['answer']])
    
    # Embed prompt and retrieve top k=5
    query_embedding = model.encode([prompt], show_progress_bar=False)
    k = 5
    distances, indices = index.search(query_embedding, k)
    retrieved_indices = indices[0]
    
    # Get retrieved documents
    retrieved_docs = [kb[idx] for idx in retrieved_indices]
    
    # Check if correct option is inside any retrieved document
    is_hit = any(correct_option_text in doc for doc in retrieved_docs)
    if is_hit:
        hits += 1

hit_rate = (hits / num_rows) * 100.0
print(f"Hits: {hits} / {num_rows}")
print(f"Hit Rate: {hit_rate:.1f}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hits: 73 / 100
Hit Rate: 73.0%


# Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
# For each row, your pipeline must do the following in order:

# Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

# Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

# Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

# Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

# Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

# What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [20]:
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

def calculate_ap3(predictions, correct_answer):
    for rank, p in enumerate(predictions[:3], start=1):
        if p == correct_answer:
            return 1.0 / rank
    return 0.0

# Load train dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')


with open('kb.pkl', 'rb') as f:
    kb = pickle.load(f)
index = faiss.read_index('kb_index.faiss')

# Load models
bi_model = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

num_rows = 20
ap_scores = []

for idx in range(num_rows):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_answer = str(row['answer'])
    
    # 1. Retrieve top 5
    query_embedding = bi_model.encode([prompt], show_progress_bar=False)
    distances, faiss_indices = index.search(query_embedding, 5)
    retrieved_indices = faiss_indices[0]
    retrieved_docs = [kb[i] for i in retrieved_indices]
    
    # 2. Rerank using Cross-Encoder
    pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)
    best_doc_idx = np.argmax(ce_scores)
    best_document = retrieved_docs[best_doc_idx]
    
    # 3. Augment prompt with the best context
    rag_string = f"Context: {best_document} Question: {prompt}"
    
    # 4. Predict probabilities of candidate labels (options A-E)
    options = {
        'A': str(row['A']),
        'B': str(row['B']),
        'C': str(row['C']),
        'D': str(row['D']),
        'E': str(row['E'])
    }
    candidate_labels = [options[letter] for letter in ['A', 'B', 'C', 'D', 'E']]
    result = zs(rag_string, candidate_labels=candidate_labels)
    
    # Match candidate labels back to original letters (A-E)
    predictions = []
    used_letters = set()
    for label in result['labels']:
        for letter in ['A', 'B', 'C', 'D', 'E']:
            if letter not in used_letters and options[letter] == label:
                predictions.append(letter)
                used_letters.add(letter)
                break
                
    # 5. Score Average Precision (AP@3)
    ap3 = calculate_ap3(predictions, correct_answer)
    ap_scores.append(ap3)

mean_map3 = np.mean(ap_scores)
print(f"Final Average MAP@3: {mean_map3:.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Final Average MAP@3: 0.975
